# Notebook auxiliar — Exploración de calidad del agua en Valmayor

> **Notebook auxiliar y exploratorio.** No forma parte del pipeline central del TFG. Se incluye como apoyo conceptual y documental para justificar que la calidad del agua es una dimensión potencialmente relevante, pero la cobertura temporal localizada es muy limitada (apenas 2006-2007), por lo que no se considera apto como variable de modelado.

A partir del informe técnico `BVCM010572.pdf`, este notebook explora proxies textuales de calidad del agua en Valmayor (turbidez, clorofila, nutrientes, etc.) y construye una **señal anual exploratoria** que se persiste en CSV. Esta señal se incorpora al merge del notebook 11 únicamente por trazabilidad y queda fuera de las variables candidatas del notebook 12.

## Objetivo

- Extraer el texto del PDF `BVCM010572.pdf` con `pdfplumber`.
- Buscar términos relacionados con calidad del agua y localizar las páginas con mayor evidencia.
- Inspeccionar manualmente las páginas relevantes.
- Construir una pequeña tabla `water_quality_signal_by_year` para los años cubiertos por el informe.
- Persistir la señal anual en CSV.

## Entradas y salidas

**Entrada:**

- `data/external/BVCM010572.pdf` — informe técnico utilizado como fuente exploratoria.

**Salida:**

- `data/external/valmayor_water_quality_signal_by_year.csv` — señal anual exploratoria (uso documental).

## Preparación del entorno

Montaje de Google Drive, importación de librerías base y comprobación de la ruta del PDF utilizado como fuente.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdfplumber

In [9]:
BASE_DIR = Path("/content/drive/MyDrive/PIDS4jj")
EXTERNAL_DIR = BASE_DIR / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

print(EXTERNAL_DIR)
print(EXTERNAL_DIR.exists())

/content/drive/MyDrive/PIDS4jj/data/external
True


In [10]:
PDF_PATH = EXTERNAL_DIR / "BVCM010572.pdf"

print(PDF_PATH)
print(PDF_PATH.exists())

/content/drive/MyDrive/PIDS4jj/data/external/BVCM010572.pdf
True


## Extracción de texto y búsqueda de términos

Se instala `pdfplumber`, se extrae el texto de todas las páginas del PDF y se busca un conjunto de palabras clave relacionadas con calidad del agua (clorofila, turbidez, nutrientes, fitoplancton, eutrofización, etc.). El resultado es un resumen de páginas en las que aparece cada término.

In [17]:
!pip install pdfplumber -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 50.8 MB/s eta 0:00:00


In [19]:
all_pages_text = []

with pdfplumber.open(PDF_PATH) as pdf:
    print("Número de páginas:", len(pdf.pages))

    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            all_pages_text.append({
                "page": i + 1,
                "text": text
            })

print("Páginas con texto extraído:", len(all_pages_text))

Número de páginas: 104
Páginas con texto extraído: 103


In [23]:
df_pdf_text = pd.DataFrame(all_pages_text)
df_pdf_text.head()

,page,text
0,1,9\nProyecto de investigación\npara la definici...
1,2,© Canal de Isabel II - 2010\nAutores:\nJ. Iñak...
2,3,Las afirmaciones recogidas en el presente docu...
3,4,Los cuadernos de I+D+i de Canal de Isabel II f...
4,5,Definición y evaluación de la aplicabilidad de...


In [24]:
full_text = "\n\n".join(df_pdf_text["text"].tolist())

print("Longitud total del texto:", len(full_text))
print(full_text[:3000])

Longitud total del texto: 290872
9
Proyecto de investigación
para la definición y evaluación
de la aplicabilidad de un bioensayo
para la determinación de la toxicidad del
agua utilizando embriones de pez cebra
www.cyii.es
Canal de Isabel II
Santa Engracia, 125. 28003 Madrid
oyasneoib
nu
ed
dadilibacilpa
al
ed
nóicaulave
y
nóicinfied
al
arap
nóicagitsevni
ed
otceyorP
i+D+I
ED
SONREDAUC
arbec
zep
ed
senoirbme
odnazilitu
auga
led
dadicixot
al
ed
nóicanimreted
al
arap
CUADERNOS DE I+D+i
9
0102/21
:nóicide
ed
ahceF
-
seralpmeje
003
:adariT

© Canal de Isabel II - 2010
Autores:
J. Iñaki Urrutia Gutiérrez
Joaquín Guinea López
Juan F. Rodríguez Plaza
Paloma Acebo País
Dirección del estudio:
J. Iñaki Urrutia Gutiérrez
Rafael Heredero Rodríguez
Edición coordinada por:
Subdirección de Comunicación y RR.PP.
CUADERNOS DE I+D+i
Expresamos nuestro agradecimiento por su colaboración y especial
aportación a Dulce Mª González Ramos y Alma Mª Jiménez Rodríguez de
la subdirección de Calidad de las Aguas y

In [25]:
keywords = [
    "valmayor",
    "clorofila",
    "clorofila a",
    "turbidez",
    "amonio",
    "amonio total",
    "ortofosfato",
    "ortofosfatos",
    "fosforo",
    "fósforo",
    "fitoplancton",
    "cianobacterias",
    "cianofíceas",
    "microcistina",
    "secchi",
    "transparencia",
    "eutrof",
    "calidad del agua"
]

keywords

['valmayor',
 'clorofila',
 'clorofila a',
 'turbidez',
 'amonio',
 'amonio total',
 'ortofosfato',
 'ortofosfatos',
 'fosforo',
 'fósforo',
 'fitoplancton',
 'cianobacterias',
 'cianofíceas',
 'microcistina',
 'secchi',
 'transparencia',
 'eutrof',
 'calidad del agua']

In [26]:
results = []

for _, row in df_pdf_text.iterrows():
    text_lower = row["text"].lower()
    for kw in keywords:
        if kw.lower() in text_lower:
            results.append({
                "page": row["page"],
                "keyword": kw
            })

df_hits = pd.DataFrame(results).drop_duplicates().sort_values(["keyword", "page"])
df_hits.head(50)

,page,keyword
27,16,amonio
42,30,amonio
64,44,amonio
69,45,amonio
73,46,amonio
82,48,amonio
126,95,amonio
4,5,calidad del agua
6,6,calidad del agua
15,10,calidad del agua


In [27]:
if len(df_hits) > 0:
    display(df_hits.groupby("keyword").agg(
        n_pages=("page", "nunique"),
        pages=("page", lambda x: sorted(list(x.unique())))
    ).sort_values("n_pages", ascending=False))
else:
    print("No se encontraron coincidencias.")

,n_pages,pages
keyword,,
microcistina,24,"[5, 7, 10, 11, 13, 16, 19, 23, 24, 51, 52, 63,..."
valmayor,21,"[7, 10, 11, 24, 26, 27, 28, 32, 34, 36, 37, 38..."
cianofíceas,12,"[5, 7, 10, 13, 15, 16, 30, 45, 50, 76, 90, 102]"
cianobacterias,9,"[5, 10, 11, 16, 48, 50, 51, 73, 78]"
fitoplancton,9,"[7, 16, 46, 48, 50, 53, 78, 85, 86]"
eutrof,8,"[30, 33, 38, 44, 46, 48, 92, 101]"
clorofila,7,"[15, 16, 42, 48, 49, 50, 95]"
amonio,7,"[16, 30, 44, 45, 46, 48, 95]"
ortofosfato,7,"[16, 30, 45, 46, 47, 48, 95]"


## Inspección de páginas relevantes

Se filtran las páginas en las que aparecen las menciones más densas de los términos de calidad del agua y se imprime su texto para revisarlo manualmente. Este paso es la base sobre la que se construye la señal anual exploratoria.

In [33]:
pages_focus = [42, 44, 46, 48, 49, 50]
df_focus = df_pdf_text[df_pdf_text["page"].isin(pages_focus)].copy()

df_focus[["page", "text"]]

,page,text
41,42,5. Evolución físico-química y biológica\nFigur...
43,44,5. Evolución físico-química y biológica\nTabla...
45,46,5. Evolución físico-química y biológica\nEl am...
47,48,5. Evolución físico-química y biológica\nFinal...
48,49,Figuras 130 y 131. Evolución de la clorofila a...
49,50,5. Evolución físico-química y biológica\n5.2.2...


In [34]:
for _, row in df_focus.iterrows():
    print("=" * 100)
    print(f"PÁGINA {row['page']}")
    print("=" * 100)
    print(row["text"][:4000])
    print("\n")

PÁGINA 42
5. Evolución físico-química y biológica
Figuras 94 y 95. Evolución del manganeso en el embalse de Valmayor, durante 2006 y 2007
Evolución del manganeso total (mg/l), 2006
Ene. Feb. Mar. Abr. May. Jun. Jul. Ago. Sept. Oct. Nov. Dic.
Evolución del manganeso total (mg/l), 2007
Ene. Feb. Mar. Abr. May. Jun. Jul. Ago. Sept. Oct. Nov. Dic.
84 85
)m(
dadidnuforP
)m(
dadidnuforP
Figuras 90 y 91. Evolución del manganeso en el embalse de Pinilla, durante 2006 y 2007
Evolución del manganeso total (mg/l), 2006
Ene. Feb. Mar. Abr. May. Jun. Jul. Ago. Sept. Oct. Nov. Dic.
Evolución del manganeso total (mg/l), 2007
Ene. Feb. Mar. Abr. May. Jun. Jul. Ago. Sept. Oct. Nov. Dic.
El manganeso sigue un patrón semejante al del hierro, siendo muy bajo en El Atazar, donde apenas supera los
100 microgramos por litro. En el resto de los embalses los máximos se localizan en el fondo, especialmente al
final de la estratificación. Son máximos más tardíos en Valmayor y Pedrezuela y más intensos en Santill

## Construcción de la señal anual exploratoria

A partir de la lectura cualitativa de las páginas relevantes se construye una pequeña tabla con una fila por año cubierto (2006, 2007), un valor `water_quality_signal` (1 = señal moderada, 2 = señal fuerte), un `evidence_type` cualitativo y notas. La tabla se persiste en `valmayor_water_quality_signal_by_year.csv`.

In [35]:
df_quality_signal = pd.DataFrame({
    "year": [2006, 2007],
    "water_quality_signal": [np.nan, np.nan],
    "evidence_type": ["", ""],
    "source": ["BVCM010572.pdf", "BVCM010572.pdf"],
    "notes": ["", ""]
})

df_quality_signal

,year,water_quality_signal,evidence_type,source,notes
0,2006,NaN,,BVCM010572.pdf,
1,2007,NaN,,BVCM010572.pdf,


In [36]:
df_quality_signal.loc[df_quality_signal["year"] == 2006, "water_quality_signal"] = 1
df_quality_signal.loc[df_quality_signal["year"] == 2006, "evidence_type"] = "moderada"
df_quality_signal.loc[df_quality_signal["year"] == 2006, "notes"] = "Menciones textuales a clorofila/turbidez/nutrientes en Valmayor"

df_quality_signal.loc[df_quality_signal["year"] == 2007, "water_quality_signal"] = 2
df_quality_signal.loc[df_quality_signal["year"] == 2007, "evidence_type"] = "fuerte"
df_quality_signal.loc[df_quality_signal["year"] == 2007, "notes"] = "Mayor evidencia textual de deterioro físico-químico/biológico en Valmayor"

df_quality_signal

,year,water_quality_signal,evidence_type,source,notes
0,2006,1.0,moderada,BVCM010572.pdf,Menciones textuales a clorofila/turbidez/nutri...
1,2007,2.0,fuerte,BVCM010572.pdf,Mayor evidencia textual de deterioro físico-qu...


In [37]:
QUALITY_SIGNAL_PATH = EXTERNAL_DIR / "valmayor_water_quality_signal_by_year.csv"

df_quality_signal.to_csv(QUALITY_SIGNAL_PATH, index=False, encoding="utf-8")

print("Guardado en:", QUALITY_SIGNAL_PATH)

Guardado en: /content/drive/MyDrive/PIDS4jj/data/external/valmayor_water_quality_signal_by_year.csv


## Conclusión

La revisión del informe técnico sugiere una **señal documental** de peor estado relativo de la calidad del agua en 2007 respecto a 2006, al aparecer una mayor intensidad de referencias a alteraciones físico-químicas y biológicas. No se trata de una medición directa, sino de un **proxy cualitativo** construido a partir del texto.

El resultado no permite afirmar una tendencia continua hasta 2025, pero sí justifica que la calidad del agua es una dimensión potencialmente relevante para el estudio de la actividad pesquera. Como **línea futura**, sería recomendable incorporar variables más completas (clorofila, turbidez, nutrientes, transparencia) cuando se disponga de una serie temporal más amplia y homogénea.

Por las limitaciones descritas, este notebook se mantiene como **auxiliar/exploratorio**: la señal generada se incluye en el merge del notebook 11 únicamente por trazabilidad y queda fuera de las variables candidatas del modelado del notebook 12.